# 04 Custom Forward: Looping Upper Layers Multiple Times

**Scenario**: Looped / recurrent-depth models run a block of layers multiple times per token, trading extra compute for stronger reasoning without adding parameters—for example, Huginn (Geiping et al. 2025) and Ouro (2025). Ouro also lets each token decide when to stop looping. Testing such architectures in a real serving engine normally requires rewriting vLLM's model forward pass and figuring out what happens to the KV cache on each iteration: write it? overwrite it? or read-only?

`@pllm.custom_forward` lets you orchestrate a token's forward pass in plain Python: model layers are exposed as range-callable functions, you write tensor math between layer spans, and a single parameter specifies KV cache semantics.

**To be clear upfront**: Qwen2.5 was not trained for looping, so repeating its layers won't make it smarter. This tutorial demonstrates the **mechanism**—the exact same code works unchanged on a recurrent-depth checkpoint from the same model family.

In [1]:
import os

os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"  # Show only vLLM errors to suppress startup logs

from dataclasses import dataclass

import torch
from transformers import AutoTokenizer
from vllm import SamplingParams

import pllm

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
CONFIG = pllm.ModelConfig(MODEL, max_model_len=512, max_num_seqs=4, max_num_batched_tokens=512,
                          gpu_memory_utilization=0.3, seed=0)

## Step 1: A Looping Workload

The `think` method receives `layers` (`pllm.model.Layers`):

- `layers()` runs the full model and returns the residual stream `taps` after every layer, where `taps[-1]` is the final layer's output;
- `layers(start=s, hidden=h)` runs from layer `s` through the last layer starting from `h`;
- The method's return tensor is what gets sent to the final norm and LM head.

Every `layers(...)` call computes K and V for those layers. The default `kv="commit"` writes them into the KV cache. Writing on every iteration of a loop overwrites the previous iteration, so later tokens see the K and V from this token's final iteration. The Ouro paper (arXiv 2510.25741, §5.4.2 Table 14) compared decode-time KV strategies for looping: storing every iteration separately scored 78.92 on GSM8K; keeping only the last iteration scored 78.85; keeping only the first iteration collapsed to 18.73. Committing on every pass corresponds to "last iteration only," whereas writing on the first pass and reading without writing afterward corresponds to "first iteration only" and breaks a loop-trained model. `kv="peek"` reads prior cached positions alongside the current pass's freshly computed K and V without writing back; it is meant for lookahead trials that leave no trace (such as speculative rollouts), not for recurrent loops here.

`loops` sets the iteration count, and `start` sets the first layer of the repeated span. To keep a non-loop-trained model from collapsing immediately, each iteration blends its output at fraction `mix` (`mix=1` is a full replacement loop). `tol` implements Ouro-style per-token early stopping: once an iteration's relative change on a token falls below `tol`, later iterations leave that token unchanged. Notice that the loop uses a fixed Python `for` bound and expresses early stopping via `torch.where` rather than `if`: control flow in forward cannot depend on tensor values, ensuring bounded per-token compute.

In [2]:
@dataclass(frozen=True)
class LatentLoop(pllm.Workload):
    start: int = 20
    loops: int = 0
    mix: float = 1.0
    tol: float = 0.0

    @pllm.custom_forward(tokens="all")
    def think(self, layers: pllm.model.Layers) -> torch.Tensor:
        hidden = layers()[-1]
        done = torch.zeros_like(hidden[:, :1], dtype=torch.bool)
        for _ in range(self.loops):
            again = layers(start=self.start, hidden=hidden)[-1]  # kv="commit": write back to KV on every pass
            step = torch.lerp(hidden, again, self.mix)
            change = (step - hidden).float().norm(dim=-1, keepdim=True) / hidden.float().norm(dim=-1, keepdim=True)
            hidden = torch.where(done, hidden, step)
            done = done | (change < self.tol)
        return hidden

## Step 2: `loops=0` Is the Original Model

With zero extra loops, `think` simply runs the whole model once. First verify that it matches plain vLLM word-for-word, confirming `custom_forward` itself alters no computation.

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)
questions = ["What is the capital of France? Answer in one sentence.", "Name three primary colors."]
prompts = [tokenizer.apply_chat_template([{"role": "user", "content": q}], tokenize=False,
                                         add_generation_prompt=True) for q in questions]
greedy = SamplingParams(max_tokens=32, temperature=0.0)

with pllm.make_llm(CONFIG) as plain:
    baseline = [output.outputs[0].text for output in plain.generate(prompts, greedy, use_tqdm=False)]


def run(workload: LatentLoop) -> list[str]:
    with pllm.make_llm(CONFIG, workload=workload) as llm:
        return [output.outputs[0].text for output in llm.generate(prompts, greedy)]


texts = run(LatentLoop(loops=0))
print("与原生 vLLM 相同:", texts == baseline)
for text in texts:
    print("   ", repr(text))

与原生 vLLM 相同: True
    'The capital of France is Paris.'
    'Three primary colors are red, blue, and yellow.'


## Step 3: Turning On the Loop

Three configurations: one pure loop from layer 20 (`mix=1`); one loop from layer 20 mixed at 10%; and three loops from layer 20 mixed at 10% with early stopping enabled (`tol=0.05`) (this 0.5B model has 24 layers, indexed 0–23).

In [4]:
for workload in (LatentLoop(loops=1, mix=1.0), LatentLoop(loops=1, mix=0.1), LatentLoop(loops=3, mix=0.1, tol=0.05)):
    print(workload)
    for text in run(workload):
        print("   ", repr(text))

LatentLoop(start=20, loops=1, mix=1.0, tol=0.0)


    'The\xa0capital of the\xa0, the\xa0, and the 1,116 other ordinary, ordinary, ordinary, ordinary, and, and, and'
    '«I, 🙂\\, would indeed indeed indeed indeed indeed indeed indeed indeed indeed indeed indeed indeed indeed indeed indeed indeed indeed,\xa0‘‘‘‘‘‘'
LatentLoop(start=20, loops=1, mix=0.1, tol=0.0)


    'The capital of France is Paris. It is the largest city in the country and the administrative center of the country. The city is known for its historical landmarks,'
    'The three primary colors are:\n\n1. Red\n2. Blue\n3. Blue\n\nThese three colors are the most common and commonly used in the field of'
LatentLoop(start=20, loops=3, mix=0.1, tol=0.05)


    'The capital of the country is indeed, and is indeed, the city of the same name, which is the capital of the country, the Republic of the Republic'
    'The three primary colors are: **Red, Blue, and Yellow.»»»»»»»»»»»»»»»»»»'


Three outcomes:

- **Pure loop** (`mix=1`) breaks a model never trained for recurrence: layers 20–23 have never seen their own outputs fed back as inputs, and subsequent tokens attend to the K and V written by that pass;
- **One loop at 10% mix**: Answers remain coherent; the first answer gets slightly longer (adding extra sentences about Paris), while the second turns yellow into a second Blue;
- **Three loops at 10% mix**: The first answer drops "France / Paris" and the second ends in repeated symbols—starting to diverge.

On a model not trained for recurrence, stronger looping causes more drift; comparing loop counts, start layers, and stopping thresholds meaningfully requires a loop-trained checkpoint, with no code changes needed here.

## Runtime Cost

Because `custom_forward` re-invokes layer spans from Python every token, vLLM's monolithic whole-model CUDA graph is no longer used, and `diagnostics().eager_reason` explains why; instead, pLLM schedules the model in segments and captures its own CUDA graphs per segment ([Example 12](../examples/12_adaptive_depth.ipynb) measures an unskipped custom forward on Qwen2.5-7B at 32 concurrent requests at 99% of native CUDA graph throughput). On multi-GPU setups, it supports tensor parallelism on vLLM 0.32.0. It only supports models whose entire KV state resides in the paged cache (dense Qwen2/Llama/Qwen3 and Qwen3-MoE; Qwen3.5's DeltaNet layers carry recurrent state outside the paged cache and are unsupported).

## What Else You Can Build

- **Feed the previous step's hidden state as the next step's input** (Coconut-style continuous thought): Declare a `pllm.PositionState`, save each position's last-layer output with `layers.save(...)`, and load it on the next token with `layers.load(...)` as `layers(hidden=...)`. See `pllm.workloads.StepLoopWorkload`;
- **Cross-layer recirculation**: Norm-match deep outputs, blend them back into shallow layers, and rerun upper layers—see `pllm.workloads.RecirculationWorkload`;
- `@pllm.custom_forward(tokens="prefill", over="span")` hands you the whole prompt span at once for Jacobi-style parallel iterations—see `pllm.workloads.JacobiRecirculationWorkload`.

## Common Mistakes

Taking full control of the forward pass comes with two easy-to-hit constraints: per-token early exit (`active=`) is restricted to decode tokens, and `pllm.hidden` cannot be read in the same workload.

In [5]:
@dataclass(frozen=True)
class ExitEverywhere(pllm.Workload):
    @pllm.custom_forward(tokens="all")  # Skip layers with active= on prompt tokens too
    def think(self, layers: pllm.model.Layers) -> torch.Tensor:
        hidden = layers()[-1]
        active = hidden.float().norm(dim=-1, keepdim=True) > 1.0
        return layers(start=20, hidden=hidden, active=active)[-1]


class LoopAndProbe(LatentLoop):
    @pllm.export(name="mlp_out", max_rows=4, max_cols=1)
    def probe(self, h: torch.Tensor = pllm.hidden(21)) -> torch.Tensor:  # Which iteration does layer 21 output belong to?
        return h.float().norm(dim=-1, keepdim=True)


for workload in (ExitEverywhere(), LoopAndProbe(loops=2)):
    try:
        pllm.explain(workload, CONFIG)
    except ValueError as error:
        print(f"{type(workload).__name__}: {error}\n")

ExitEverywhere: Use @pllm.custom_forward(tokens="decode") for active=: skipped prompt tokens would leave approximate K/V that every later token reads. Prompt rows run the full model by default; add an unmasked tokens="prefill" method only to change prompt computation (prefer over="span").

LoopAndProbe: probe reads block or model outputs, which a @pllm.custom_forward does not produce; use pllm.residual or pllm.layer_logits to read each layer's last visited stream



**Why `active=` is decode-only.** Frozen rows skip later layers, so K and V for those skipped layers can only be approximated. Skipping layers on a single generated token affects only a few later positions; skipping them on prompt tokens corrupts K and V that every subsequent token will attend to. Prompts therefore run the full model by default.

**Why `pllm.hidden` is disallowed.** Inside a loop, layer 21 may execute multiple times per token—or zero times when skipping—so "layer 21's output" no longer has a single well-defined meaning. `pllm.residual(layer)` and `pllm.layer_logits(...)` read the residual stream from the **last visited** pass at that layer, which is unambiguous and works alongside `custom_forward`.

In addition, loop counts must be static Python `for` loops, and early stopping must use `torch.where` and boolean masks rather than `if`, `break`, or `while`: the forward pass is captured into CUDA graphs, so the sequence of executed ops at each step must be fixed ahead of time.

## Self-Check

1. `layers(...)` defaults to `kv="commit"`. If pass 1 uses `commit` and subsequent passes use `kv="peek"`, which pass's K and V do later tokens read?

<details><summary>Answer</summary>

Pass 1's. `peek` reads without writing, leaving pass 1's K and V in the cache—corresponding to the "first iteration only" strategy in the Ouro paper, which severely degrades loop-trained models.
</details>

2. What does matching plain vLLM identically at `loops=0` demonstrate?

<details><summary>Answer</summary>

That `custom_forward` itself—wrapping layers into callable spans and scheduling them in segments via pLLM—alters no computation; any change when looping comes purely from your loop logic.
</details>

3. With `custom_forward`, `eager_reason` is non-empty. Does that mean the model falls back to ungraphed op-by-op execution at half speed?

<details><summary>Answer</summary>

No. vLLM's own whole-model CUDA graph is bypassed, but pLLM captures its own graphs for each layer segment. In [12 Adaptive Depth](../examples/12_adaptive_depth.ipynb), a non-skipping custom forward on Qwen2.5-7B with 32 concurrent requests achieves ~99% of native CUDA graph throughput.
</details>

Next up, [05 Feedback Loop](05_host_feedback.ipynb) returns to training: closing the loop of generation, majority voting, and weight updates inside a single engine.